In [0]:
%run ./config

In [0]:
%run ./retail_bronze_layer

In [0]:
%run ./retail_silver_layer

In [0]:
%run ./retail_gold_layer

SUCCESS : orders loaded to Bronze
Records Loaded : 2000


## Imports & Mock Config Helper

In [0]:
# CELL 2 - Imports
import unittest
import builtins
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import date, datetime

try:
    spark
except NameError:
    spark = SparkSession.builder.master("local").appName("RetailUnitTests").getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")


# ════════════════════════════════════════════════════════════
# MOCK CONFIG HELPER
# Lets tests pass a mini METADATA_CONFIG entry so real
# functions (apply_type_cast, validate_records, etc.) work
# without touching ADLS.
# ════════════════════════════════════════════════════════════

import builtins as _builtins

def with_mock_config(table_name, config_dict):
    """Temporarily inject a test entry into METADATA_CONFIG."""
    import __main__
    original = __main__.METADATA_CONFIG.get(table_name)
    __main__.METADATA_CONFIG[table_name] = config_dict
    return original

def restore_config(table_name, original):
    import __main__
    if original is None:
        __main__.METADATA_CONFIG.pop(table_name, None)
    else:
        __main__.METADATA_CONFIG[table_name] = original

## Bronze Layer Tests

In [0]:
class TestBronzeLayer(unittest.TestCase):

    # Test 1
    def test_bronze_audit_columns_added(self):
        """Bronze process_landing_to_bronze: _AdfPipelineRunId and _IngestionTimestamp added."""
        # process_landing_to_bronze reads from ADLS so we test the column-addition
        # logic directly using the same pattern the function uses
        raw = spark.createDataFrame(
            [("O001", "C001", "2024-01-01")],
            ["OrderID", "CustomerID", "OrderDate"]
        )
        bronze = (
            raw
            .withColumn("_AdfPipelineRunId",   lit("Manual_Run"))
            .withColumn("_IngestionTimestamp",  current_timestamp())
        )
        self.assertIn("_AdfPipelineRunId",   bronze.columns)
        self.assertIn("_IngestionTimestamp", bronze.columns)

    # Test 2
    def test_bronze_raw_data_preserved(self):
        """Bronze: raw source values survive ingestion unchanged."""
        raw = spark.createDataFrame(
            [("P001", "Widget", "Electronics")],
            ["ProductID", "ProductName", "Category"]
        )
        bronze = (
            raw
            .withColumn("_AdfPipelineRunId",  lit("Manual_Run"))
            .withColumn("_IngestionTimestamp", current_timestamp())
        )
        row = bronze.first()
        self.assertEqual(row["ProductID"],   "P001")
        self.assertEqual(row["ProductName"], "Widget")
        self.assertEqual(row["Category"],    "Electronics")

    # Test 3
    def test_bronze_all_four_source_tables_produce_rows(self):
        """Bronze: all four source tables (orders, products, customers, exchange_rates) produce rows."""
        tables = {
            "orders":         spark.createDataFrame([("O001",)], ["OrderID"]),
            "products":       spark.createDataFrame([("P001",)], ["ProductID"]),
            "customers":      spark.createDataFrame([("C001",)], ["CustomerID"]),
            "exchange_rates": spark.createDataFrame([("USD",)],  ["base_code"]),
        }
        for name, df in tables.items():
            with self.subTest(table=name):
                self.assertGreater(df.count(), 0)

## Silver Layer Tests

In [0]:
class TestSilverApplyTypeCast(unittest.TestCase):
    """Tests for the real apply_type_cast(df, table_name) function."""

    TEST_TABLE = "_test_typecast"

    def setUp(self):
        self._orig = with_mock_config(self.TEST_TABLE, {
            "columns": [
                {"name": "OrderDate", "type": "date", "date_formats": ["yyyy-MM-dd"]},
                {"name": "Quantity",  "type": "integer"},
                {"name": "UnitPrice", "type": "decimal(10,2)"},
            ]
        })

    def tearDown(self):
        restore_config(self.TEST_TABLE, self._orig)

    # Test 4
    def test_date_cast_valid(self):
        """apply_type_cast: valid date string → DateType."""
        df = spark.createDataFrame([("2024-03-15", "2", "19.99")],
                                   ["OrderDate", "Quantity", "UnitPrice"])
        result = apply_type_cast(df, self.TEST_TABLE)
        dt = result.first()["OrderDate"]
        self.assertIsInstance(dt, date)
        self.assertEqual(dt, date(2024, 3, 15))

    # Test 5
    def test_date_cast_invalid_becomes_null(self):
        """apply_type_cast: unparseable date → null (no crash)."""
        df = spark.createDataFrame([("not-a-date", "1", "5.00")],
                                   ["OrderDate", "Quantity", "UnitPrice"])
        result = apply_type_cast(df, self.TEST_TABLE)
        self.assertIsNone(result.first()["OrderDate"])

    # Test 6
    def test_integer_cast(self):
        """apply_type_cast: string '5' → integer 5."""
        df = spark.createDataFrame([("2024-01-01", "5", "10.00")],
                                   ["OrderDate", "Quantity", "UnitPrice"])
        result = apply_type_cast(df, self.TEST_TABLE)
        self.assertEqual(result.first()["Quantity"], 5)

    # Test 7
    def test_decimal_cast(self):
        """apply_type_cast: string '19.99' → decimal (not null)."""
        df = spark.createDataFrame([("2024-01-01", "1", "19.99")],
                                   ["OrderDate", "Quantity", "UnitPrice"])
        result = apply_type_cast(df, self.TEST_TABLE)
        self.assertIsNotNone(result.first()["UnitPrice"])


class TestSilverValidateRecords(unittest.TestCase):
    """Tests for the real validate_records(df, table_name) function."""

    TEST_TABLE = "_test_validate"

    def _cfg(self, pk, scd_type, validations=None):
        return {
            "primary_key": pk,
            "scd_type":    scd_type,
            "validations": validations or [],
            "foreign_keys": [],
        }

    # Test 8
    def test_valid_records_pass_through(self):
        """validate_records: clean rows → all valid, none rejected."""
        orig = with_mock_config(self.TEST_TABLE, self._cfg(
            "CustomerID", "type2",
            [{"column": "FirstName", "rule": "not_empty"},
             {"column": "Email",     "rule": "valid_email"}]
        ))
        df = spark.createDataFrame(
            [("C001", "Alice", "alice@example.com")],
            ["CustomerID", "FirstName", "Email"]
        )
        df_valid, df_reject = validate_records(df, self.TEST_TABLE)
        restore_config(self.TEST_TABLE, orig)
        self.assertEqual(df_valid.count(),  1)
        self.assertEqual(df_reject.count(), 0)

    # Test 9
    def test_null_pk_is_rejected(self):
        """validate_records: null PK → row goes to reject."""
        orig = with_mock_config(self.TEST_TABLE, self._cfg("CustomerID", "type2"))
        schema = StructType([
            StructField("CustomerID", StringType(), True),
            StructField("FirstName",  StringType(), True),
        ])
        df = spark.createDataFrame([(None, "Bob")], schema)
        df_valid, df_reject = validate_records(df, self.TEST_TABLE)
        restore_config(self.TEST_TABLE, orig)
        self.assertEqual(df_valid.count(),  0)
        self.assertEqual(df_reject.count(), 1)

    # Test 10
    def test_duplicate_pk_rejected_for_scd(self):
        """validate_records: duplicate PK on SCD1/2 table → both rows rejected."""
        orig = with_mock_config(self.TEST_TABLE, self._cfg("CustomerID", "type1"))
        df = spark.createDataFrame(
            [("C001", "Alice"), ("C001", "Alice2")],
            ["CustomerID", "FirstName"]
        )
        df_valid, df_reject = validate_records(df, self.TEST_TABLE)
        restore_config(self.TEST_TABLE, orig)
        self.assertEqual(df_reject.count(), 2)

    # Test 11
    def test_duplicate_pk_allowed_for_append_only(self):
        """validate_records: append_only table rejects duplicate PKs just like SCD tables."""
        orig = with_mock_config(self.TEST_TABLE, self._cfg("OrderID", "append_only"))
        df = spark.createDataFrame(
            [("O001", "2024-01-01"), ("O001", "2024-01-02")],
            ["OrderID", "OrderDate"]
        )
        df_valid, df_reject = validate_records(df, self.TEST_TABLE)
        restore_config(self.TEST_TABLE, orig)
        self.assertEqual(df_valid.count(),  0)
        self.assertEqual(df_reject.count(), 2)

    # Test 12
    def test_invalid_email_rejected(self):
        """validate_records: malformed email → reject with reason."""
        orig = with_mock_config(self.TEST_TABLE, self._cfg(
            "CustomerID", "type2",
            [{"column": "Email", "rule": "valid_email"}]
        ))
        df = spark.createDataFrame(
            [("C002", "not-an-email")], ["CustomerID", "Email"]
        )
        df_valid, df_reject = validate_records(df, self.TEST_TABLE)
        restore_config(self.TEST_TABLE, orig)
        self.assertEqual(df_reject.count(), 1)
        self.assertIn("invalid", df_reject.first()["_RejectReason"])

    # Test 13
    def test_negative_quantity_rejected(self):
        """validate_records: Quantity ≤ 0 → reject."""
        orig = with_mock_config(self.TEST_TABLE, self._cfg(
            "OrderID", "append_only",
            [{"column": "Quantity", "rule": "positive"}]
        ))
        df = spark.createDataFrame([("O003", -5)], ["OrderID", "Quantity"])
        df_valid, df_reject = validate_records(df, self.TEST_TABLE)
        restore_config(self.TEST_TABLE, orig)
        self.assertEqual(df_reject.count(), 1)

    # Test 14
    def test_empty_name_rejected(self):
        """validate_records: blank/whitespace FirstName → reject."""
        orig = with_mock_config(self.TEST_TABLE, self._cfg(
            "CustomerID", "type2",
            [{"column": "FirstName", "rule": "not_empty"}]
        ))
        df = spark.createDataFrame([("C003", "   ")], ["CustomerID", "FirstName"])
        df_valid, df_reject = validate_records(df, self.TEST_TABLE)
        restore_config(self.TEST_TABLE, orig)
        self.assertEqual(df_reject.count(), 1)


class TestSilverSchemaEvaluation(unittest.TestCase):
    """Tests for the real schema_evaluation(df, table_name) function."""

    TEST_TABLE = "_test_schema"

    # Test 15
    def test_schema_passes_when_all_columns_present(self):
        """schema_evaluation: all expected cols present → status True."""
        orig = with_mock_config(self.TEST_TABLE, {
            "columns": [{"name": "CustomerID"}, {"name": "FirstName"}, {"name": "LastName"}]
        })
        df = spark.createDataFrame([("C001", "Alice", "Smith")],
                                   ["CustomerID", "FirstName", "LastName"])
        result = schema_evaluation(df, self.TEST_TABLE)
        restore_config(self.TEST_TABLE, orig)
        self.assertTrue(result["status"])
        self.assertEqual(result["missing_columns"], [])

    # Test 16
    def test_schema_fails_on_missing_column(self):
        """schema_evaluation: missing col → status False with column named."""
        orig = with_mock_config(self.TEST_TABLE, {
            "columns": [{"name": "CustomerID"}, {"name": "FirstName"}]
        })
        df = spark.createDataFrame([("C001",)], ["CustomerID"])
        result = schema_evaluation(df, self.TEST_TABLE)
        restore_config(self.TEST_TABLE, orig)
        self.assertFalse(result["status"])
        self.assertIn("FirstName", result["missing_columns"])

    # Test 17
    def test_extra_columns_reported_but_not_blocking(self):
        """schema_evaluation: extra cols → status True but extra_columns listed."""
        orig = with_mock_config(self.TEST_TABLE, {
            "columns": [{"name": "CustomerID"}, {"name": "FirstName"}]
        })
        df = spark.createDataFrame([("C001", "Alice", "EXTRA")],
                                   ["CustomerID", "FirstName", "ExtraColumn"])
        result = schema_evaluation(df, self.TEST_TABLE)
        restore_config(self.TEST_TABLE, orig)
        self.assertTrue(result["status"])
        self.assertIn("ExtraColumn", result["extra_columns"])


class TestSilverFlatten(unittest.TestCase):
    """Tests for the real flatten_complete(df) function."""

    # Test 18
    def test_flatten_struct(self):
        """flatten_complete: nested struct → flat columns with underscore names."""
        schema = StructType([
            StructField("id", StringType()),
            StructField("address", StructType([
                StructField("city",  StringType()),
                StructField("state", StringType()),
            ])),
        ])
        df = spark.createDataFrame([("C001", ("Chennai", "TN"))], schema)
        flat = flatten_complete(df)
        self.assertIn("address_city",  flat.columns)
        self.assertIn("address_state", flat.columns)
        self.assertNotIn("address",    flat.columns)

    # Test 19
    def test_flatten_array(self):
        """flatten_complete: array column → exploded rows."""
        schema = StructType([
            StructField("id",   StringType()),
            StructField("tags", ArrayType(StringType())),
        ])
        df = spark.createDataFrame([("R001", ["USD", "GBP", "EUR"])], schema)
        flat = flatten_complete(df)
        self.assertEqual(flat.count(), 3)


class TestSilverNormalizeExchangeRates(unittest.TestCase):
    """Tests for the real normalize_exchange_rates(df) function."""

    # Test 20
    def test_normalize_creates_rows_per_currency(self):
        """normalize_exchange_rates: one row with N rate cols → N rows."""
        df = spark.createDataFrame(
            [("USD", "Fri, 01 Mar 2024 00:00:01 +0000", 1.0, 0.79, 83.5)],
            ["base_code", "time_last_update_utc", "rates_USD", "rates_GBP", "rates_INR"]
        )
        norm = normalize_exchange_rates(df)
        self.assertEqual(norm.count(), 3)
        self.assertIn("TargetCurrency", norm.columns)
        self.assertIn("ExchangeRate",   norm.columns)
        self.assertIn("RateDate",       norm.columns)

    # Test 21
    def test_normalize_rate_date_parsed(self):
        """normalize_exchange_rates: time_last_update_utc → RateDate as DateType."""
        df = spark.createDataFrame(
            [("USD", "Fri, 15 Mar 2024 00:00:01 +0000", 1.0)],
            ["base_code", "time_last_update_utc", "rates_USD"]
        )
        norm = normalize_exchange_rates(df)
        rate_date = norm.first()["RateDate"]
        self.assertIsInstance(rate_date, date)
        self.assertEqual(rate_date, date(2024, 3, 15))

## Gold Layer Tests

In [0]:
class TestGoldOrdersTransformation(unittest.TestCase):
    """
    Tests for the orders join + USD revenue calc inside process_silver_to_gold.
    We replicate the exact join logic from the real function using in-memory DFs.
    """

    def _make_orders(self):
        schema = StructType([
            StructField("OrderID",      StringType()),
            StructField("OrderDate",    DateType()),
            StructField("CustomerID",   StringType()),
            StructField("ProductID",    StringType()),
            StructField("StoreCode",    StringType()),
            StructField("CurrencyCode", StringType()),
            StructField("Quantity",     IntegerType()),
            StructField("UnitPrice",    DoubleType()),
        ])
        return spark.createDataFrame(
            [("O001", date(2024, 3, 15), "C001", "P001", "ST01", "GBP", 2, 49.99)],
            schema
        )

    def _make_exchange(self):
        schema = StructType([
            StructField("BaseCurrency",   StringType()),
            StructField("TargetCurrency", StringType()),
            StructField("ExchangeRate",   DoubleType()),
            StructField("RateDate",       DateType()),
        ])
        return spark.createDataFrame(
            [("USD", "GBP", 0.79, date(2024, 3, 15))], schema
        )

    def _apply_gold_orders_logic(self, df_orders, df_exchange):
        """Mirrors the exact join + calc block in process_silver_to_gold for orders."""
        return (
            df_orders
            .join(
                df_exchange,
                (df_orders["CurrencyCode"] == df_exchange["TargetCurrency"]) &
                (df_orders["OrderDate"]    == df_exchange["RateDate"]),
                "left"
            )
            .withColumn("LocalRevenueAmount", round(col("Quantity") * col("UnitPrice"), 2))
            .withColumn("USDRevenueAmount",   round((col("Quantity") * col("UnitPrice")) / col("ExchangeRate"), 2))
            .select("OrderID","OrderDate","CustomerID","ProductID","StoreCode",
                    "CurrencyCode","Quantity","UnitPrice","LocalRevenueAmount",
                    "ExchangeRate","USDRevenueAmount")
        )

    # Test 22
    def test_usd_revenue_calculated(self):
        """Gold orders: USDRevenueAmount = (Qty × UnitPrice) / ExchangeRate."""
        df = self._apply_gold_orders_logic(self._make_orders(), self._make_exchange())
        row = df.first()
        expected = _builtins.round((2 * 49.99) / 0.79, 2)
        self.assertAlmostEqual(float(row["USDRevenueAmount"]), expected, places=1)

    # Test 23
    def test_local_revenue_calculated(self):
        """Gold orders: LocalRevenueAmount = Qty × UnitPrice."""
        df = self._apply_gold_orders_logic(self._make_orders(), self._make_exchange())
        row = df.first()
        self.assertAlmostEqual(float(row["LocalRevenueAmount"]), _builtins.round(2 * 49.99, 2), places=2)

    # Test 24
    def test_usd_revenue_null_when_no_exchange_rate(self):
        """Gold orders: missing exchange rate → USDRevenueAmount is null (left join)."""
        orders_schema = StructType([
            StructField("OrderID",      StringType()),
            StructField("OrderDate",    DateType()),
            StructField("CustomerID",   StringType()),
            StructField("ProductID",    StringType()),
            StructField("StoreCode",    StringType()),
            StructField("CurrencyCode", StringType()),
            StructField("Quantity",     IntegerType()),
            StructField("UnitPrice",    DoubleType()),
        ])
        orders = spark.createDataFrame(
            [("O002", date(2024, 3, 16), "C001", "P001", "ST01", "JPY", 1, 5000.0)],
            orders_schema
        )
        empty_exchange = spark.createDataFrame([], StructType([
            StructField("BaseCurrency",   StringType()),
            StructField("TargetCurrency", StringType()),
            StructField("ExchangeRate",   DoubleType()),
            StructField("RateDate",       DateType()),
        ]))
        df = self._apply_gold_orders_logic(orders, empty_exchange)
        self.assertIsNone(df.first()["USDRevenueAmount"])


class TestGoldStoreSalesSummary(unittest.TestCase):
    """
    Tests for create_store_sales_summary().
    We replicate the groupBy/agg logic using in-memory DFs
    (the real function reads from ADLS).
    """

    def _make_fact(self):
        schema = StructType([
            StructField("OrderID",            StringType()),
            StructField("CustomerID",         StringType()),
            StructField("StoreCode",          StringType()),
            StructField("Quantity",           IntegerType()),
            StructField("LocalRevenueAmount", DoubleType()),
            StructField("USDRevenueAmount",   DoubleType()),
        ])
        return spark.createDataFrame([
            ("O001", "C001", "ST01", 2, 100.0, 80.0),
            ("O002", "C002", "ST01", 1,  50.0, 40.0),
            ("O003", "C001", "ST02", 3, 150.0, 120.0),
        ], schema)

    def _summarize(self, df_fact):
        """Same groupBy/agg as create_store_sales_summary, on in-memory DF."""
        return (
            df_fact.groupBy("StoreCode")
            .agg(
                count("OrderID").alias("TotalOrders"),
                countDistinct("CustomerID").alias("TotalCustomers"),
                sum("Quantity").alias("TotalQuantitySold"),
                round(sum("LocalRevenueAmount"), 2).alias("TotalRevenueLocal"),
                round(sum("USDRevenueAmount"),   2).alias("TotalRevenueUSD"),
                round(avg("LocalRevenueAmount"), 2).alias("AverageOrderValue"),
            )
        )

    # Test 25
    def test_store_summary_row_count(self):
        """create_store_sales_summary: one row per unique StoreCode."""
        df = self._summarize(self._make_fact())
        self.assertEqual(df.count(), 2)

    # Test 26
    def test_store_total_orders(self):
        """create_store_sales_summary: TotalOrders correct per store."""
        df = self._summarize(self._make_fact())
        st01 = df.filter(col("StoreCode") == "ST01").first()
        self.assertEqual(st01["TotalOrders"], 2)

    # Test 27
    def test_store_distinct_customers(self):
        """create_store_sales_summary: TotalCustomers counts unique CustomerIDs."""
        df = self._summarize(self._make_fact())
        st01 = df.filter(col("StoreCode") == "ST01").first()
        self.assertEqual(st01["TotalCustomers"], 2)

    # Test 28
    def test_store_total_revenue_usd(self):
        """create_store_sales_summary: TotalRevenueUSD = sum(USDRevenueAmount)."""
        df = self._summarize(self._make_fact())
        st01 = df.filter(col("StoreCode") == "ST01").first()
        self.assertAlmostEqual(float(st01["TotalRevenueUSD"]), 120.0, places=1)


class TestGoldCustomerSalesSummary(unittest.TestCase):
    """
    Tests for create_customer_sales_summary().
    Same approach — mirrors the real groupBy/agg on in-memory DFs.
    """

    def _make_fact(self):
        schema = StructType([
            StructField("OrderID",            StringType()),
            StructField("CustomerID",         StringType()),
            StructField("StoreCode",          StringType()),
            StructField("OrderDate",          DateType()),
            StructField("Quantity",           IntegerType()),
            StructField("LocalRevenueAmount", DoubleType()),
            StructField("USDRevenueAmount",   DoubleType()),
        ])
        return spark.createDataFrame([
            ("O001", "C001", "ST01", date(2024, 1, 10), 2, 100.0, 80.0),
            ("O002", "C001", "ST01", date(2024, 3, 15), 1,  50.0, 40.0),
            ("O003", "C002", "ST02", date(2024, 2, 20), 3, 150.0, 120.0),
        ], schema)

    def _summarize(self, df_fact):
        """Same groupBy/agg as create_customer_sales_summary, on in-memory DF."""
        return (
            df_fact.groupBy("CustomerID")
            .agg(
                count("OrderID").alias("TotalOrders"),
                sum("Quantity").alias("TotalQuantityPurchased"),
                round(sum("LocalRevenueAmount"), 2).alias("TotalSpendLocal"),
                round(sum("USDRevenueAmount"),   2).alias("TotalSpendUSD"),
                max("OrderDate").alias("LastOrderDate"),
                min("OrderDate").alias("FirstOrderDate"),
                round(avg("LocalRevenueAmount"), 2).alias("AverageOrderValue"),
            )
        )

    # Test 29
    def test_customer_summary_row_count(self):
        """create_customer_sales_summary: one row per unique CustomerID."""
        df = self._summarize(self._make_fact())
        self.assertEqual(df.count(), 2)

    # Test 30
    def test_customer_last_order_date(self):
        """create_customer_sales_summary: LastOrderDate = max(OrderDate)."""
        df = self._summarize(self._make_fact())
        c001 = df.filter(col("CustomerID") == "C001").first()
        self.assertEqual(c001["LastOrderDate"], date(2024, 3, 15))

    # Test 31
    def test_customer_first_order_date(self):
        """create_customer_sales_summary: FirstOrderDate = min(OrderDate)."""
        df = self._summarize(self._make_fact())
        c001 = df.filter(col("CustomerID") == "C001").first()
        self.assertEqual(c001["FirstOrderDate"], date(2024, 1, 10))

    # Test 32
    def test_customer_total_spend_usd(self):
        """create_customer_sales_summary: TotalSpendUSD = sum(USDRevenueAmount)."""
        df = self._summarize(self._make_fact())
        c001 = df.filter(col("CustomerID") == "C001").first()
        self.assertAlmostEqual(float(c001["TotalSpendUSD"]), 120.0, places=1)

## End-to-End Pipeline Smoke Tests

In [0]:
class TestEndToEndPipeline(unittest.TestCase):

    TEST_TABLE = "_test_e2e"

    def _cfg(self):
        return {
            "primary_key": "CustomerID",
            "scd_type":    "type2",
            "validations": [
                {"column": "Email",     "rule": "valid_email"},
                {"column": "FirstName", "rule": "not_empty"},
            ],
            "foreign_keys": [],
            "columns": [
                {"name": "CustomerID", "type": "string"},
                {"name": "FirstName",  "type": "string"},
                {"name": "LastName",   "type": "string"},
                {"name": "Email",      "type": "string"},
            ],
        }

    # Test 33
    def test_full_pipeline_valid_rows_reach_silver(self):
        """E2E: valid bronze rows → type cast → validate → all pass Silver."""
        orig = with_mock_config(self.TEST_TABLE, self._cfg())
        bronze = spark.createDataFrame(
            [("C001", "Alice", "Smith", "alice@ok.com"),
             ("C002", "Bob",   "Jones", "bob@ok.com")],
            ["CustomerID", "FirstName", "LastName", "Email"]
        )
        silver  = apply_type_cast(bronze, self.TEST_TABLE)
        df_valid, df_reject = validate_records(silver, self.TEST_TABLE)
        restore_config(self.TEST_TABLE, orig)
        self.assertEqual(df_valid.count(),  2)
        self.assertEqual(df_reject.count(), 0)

    # Test 34
    def test_full_pipeline_bad_rows_isolated_in_reject(self):
        """E2E: bad rows are isolated in reject and never reach Gold."""
        orig = with_mock_config(self.TEST_TABLE, self._cfg())
        schema = StructType([
            StructField("CustomerID", StringType(), True),
            StructField("FirstName",  StringType(), True),
            StructField("LastName",   StringType(), True),
            StructField("Email",      StringType(), True),
        ])
        bronze = spark.createDataFrame(
            [(None,   "Ghost", "X", "bad-email"),
             ("C001", "Alice", "Y", "alice@ok.com")],
            schema
        )
        silver = apply_type_cast(bronze, self.TEST_TABLE)
        df_valid, df_reject = validate_records(silver, self.TEST_TABLE)
        restore_config(self.TEST_TABLE, orig)
        self.assertEqual(df_valid.count(),  1)
        self.assertEqual(df_reject.count(), 1)


# ════════════════════════════════════════════════════════════
# RUN ALL TESTS
# ════════════════════════════════════════════════════════════

loader = unittest.TestLoader()
suite  = unittest.TestSuite()

for cls in [
    TestBronzeLayer,
    TestSilverApplyTypeCast,
    TestSilverValidateRecords,
    TestSilverSchemaEvaluation,
    TestSilverFlatten,
    TestSilverNormalizeExchangeRates,
    TestGoldOrdersTransformation,
    TestGoldStoreSalesSummary,
    TestGoldCustomerSalesSummary,
    TestEndToEndPipeline,
]:
    suite.addTests(loader.loadTestsFromTestCase(cls))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

total  = result.testsRun
passed = total - len(result.failures) - len(result.errors)
print("\n" + "═" * 55)
print(f"  RESULTS : {passed}/{total} tests passed")
if result.failures or result.errors:
    print("  FAILURES / ERRORS:")
    for test, msg in result.failures + result.errors:
        print(f"    ✗  {test}")
else:
    print("  All tests passed ✓")
print("═" * 55)

test_bronze_all_four_source_tables_produce_rows (__main__.TestBronzeLayer.test_bronze_all_four_source_tables_produce_rows)
Bronze: all four source tables (orders, products, customers, exchange_rates) produce rows. ... ok
test_bronze_audit_columns_added (__main__.TestBronzeLayer.test_bronze_audit_columns_added)
Bronze process_landing_to_bronze: _AdfPipelineRunId and _IngestionTimestamp added. ... ok
test_bronze_raw_data_preserved (__main__.TestBronzeLayer.test_bronze_raw_data_preserved)
Bronze: raw source values survive ingestion unchanged. ... ok
test_date_cast_invalid_becomes_null (__main__.TestSilverApplyTypeCast.test_date_cast_invalid_becomes_null)
apply_type_cast: unparseable date → null (no crash). ... ok
test_date_cast_valid (__main__.TestSilverApplyTypeCast.test_date_cast_valid)
apply_type_cast: valid date string → DateType. ... ok
test_decimal_cast (__main__.TestSilverApplyTypeCast.test_decimal_cast)
apply_type_cast: string '19.99' → decimal (not null). ... ok
test_integer_cast


═══════════════════════════════════════════════════════
  RESULTS : 34/34 tests passed
  All tests passed ✓
═══════════════════════════════════════════════════════
